To match a query to a list of news articles, we can leverage a Graph Attention Network (GAT) by treating both the query and news articles as nodes in a graph. Each article can be represented as a node with features like word embeddings (e.g., BERT embeddings, TF-IDF, etc.), and edges between nodes can capture relationships between the query and articles, such as relevance or similarity. We can then train a GAT to determine which articles are most relevant to a given query.
Problem Breakdown:

    Entities: The nodes in the graph are:
        The query (a node).
        A set of news articles (nodes).

    Edges: Edges represent relationships between the query and the articles. This could be:
        Semantic Similarity: Articles that are semantically close to the query would have stronger edges.
        Relevance: The edge weights could be based on how relevant the article is to the query.

    Goal: Given a query, the task is to use the trained GAT model to find the most relevant news articles from the list.

High-Level Steps:

    Preprocess Data:
        Convert each article and query to vector representations (embeddings).
        Create a graph with nodes representing the query and articles.
        Create edges between the query and articles based on similarity or relevance.
    Train a GAT Model:
        Use a GAT model to learn which articles are most relevant to the query by leveraging the graph structure.
    Inference:
        After training, use the model to infer the relevance of articles for a new query.

pip install torch torch-geometric transformers


In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Function to get BERT embeddings for a text
def get_bert_embeddings(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    outputs = bert_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)  # Average pooling over all tokens
    return embeddings

# Example query and articles
query = "How does climate change affect global agriculture?"
articles = [
    "Climate change is having significant effects on agriculture worldwide.",
    "New technology is transforming the farming industry.",
    "The role of artificial intelligence in climate change mitigation.",
    "Global warming and its impact on crop yields."
]

# Get BERT embeddings for the query and articles
query_embedding = get_bert_embeddings(query)
article_embeddings = [get_bert_embeddings(article) for article in articles]


3. Create Graph Data

For simplicity, we will create a graph where:

    The query node is connected to each article node.
    We will calculate a similarity score (e.g., cosine similarity) between the query and each article, which will define the edge weights.

In [ ]:
import torch
from torch_geometric.data import Data
from sklearn.metrics.pairwise import cosine_similarity

# Function to calculate cosine similarity
def cosine_sim(x, y):
    return cosine_similarity(x.cpu().numpy(), y.cpu().numpy())

# Compute cosine similarities between query and articles
similarities = [cosine_sim(query_embedding, article_embedding)[0][0] for article_embedding in article_embeddings]

# Convert similarities to tensor (edge weights)
edge_weights = torch.tensor(similarities, dtype=torch.float)

# Create edge_index representing connections between the query node (node 0) and the articles (nodes 1, 2, 3, ...)
edge_index = torch.tensor([
    [0, 0, 0, 0],  # Query node is connected to all article nodes
    [1, 2, 3, 4]   # Article nodes (1, 2, 3, 4)
], dtype=torch.long)

# Combine the query and article embeddings into a single tensor (node features)
node_features = torch.cat([query_embedding, torch.cat(article_embeddings)], dim=0)

# Create the graph data object
data = Data(x=node_features, edge_index=edge_index, edge_attr=edge_weights)


4. Define GAT Model

We’ll now define a Graph Attention Network (GAT) for the task. The model will use attention to focus on the most relevant articles given the query.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class GATModel(nn.Module):
    def __init__(self, in_channels, out_channels, num_heads=2):
        super(GATModel, self).__init__()
        
        # First GAT layer with multi-head attention
        self.gat1 = GATConv(in_channels, 8, heads=num_heads, dropout=0.6)
        
        # Second GAT layer to output final node embeddings
        self.gat2 = GATConv(8 * num_heads, out_channels, heads=1, dropout=0.6)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        x = F.elu(self.gat1(x, edge_index, edge_attr))  # First GAT layer with ELU activation
        x = self.gat2(x, edge_index, edge_attr)  # Second GAT layer
        return x  # Output node embeddings


5. Train the GAT Model

We will train the GAT model using node classification as a proxy task, where the query is treated as the target node. The objective is to classify which articles are relevant (based on similarity).

In [ ]:
# Define the optimizer and loss function
model = GATModel(in_channels=768, out_channels=2, num_heads=2)  # BERT embeddings are of size 768
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

# Dummy labels for node classification (we assume node 0 is the query, and we label articles as relevant/irrelevant)
labels = torch.tensor([1, 0, 1, 1, 0], dtype=torch.long)  # 1 for relevant, 0 for irrelevant

# Training loop
num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    # Forward pass
    output = model(data)
    
    # Loss for the query node (node 0)
    loss = criterion(output[0].unsqueeze(0), labels[0].unsqueeze(0))  # Query node is the first node
    
    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')


6. Inference: Match Query to Articles

After training, we can infer which articles are most relevant to the query based on the GAT model’s node embeddings.

In [ ]:
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    output = model(data)  # Forward pass to get embeddings

# Get the relevance scores for articles (nodes 1, 2, 3, 4)
article_scores = output[1:].max(dim=1)[0]  # Get the scores for the article nodes (excluding the query node)
print("Relevance scores for articles:", article_scores)


The article_scores will represent the relevance of each article to the query. The higher the score, the more relevant the article is.
Conclusion

This approach allows us to match a query to a list of articles by:

    Embedding both the query and the articles into a shared vector space using a pre-trained model like BERT.
    Building a graph where each article and the query are connected by edges representing semantic similarities.
    Training a GAT to learn attention mechanisms that focus on the most relevant articles to the query.
    Using the learned embeddings and attention scores to infer which articles are most relevant.

By leveraging the attention mechanism, the model can dynamically adjust the importance of different articles based on the query, making it effective for tasks like content recommendation, question answering, and semantic search.